In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [1]:
# Connect to your Weaviate instance(s).
# Set TGT_URL = SRC_URL to migrate within the same cluster.

import weaviate
import weaviate.classes as wvc
from weaviate.classes.init import Auth

# --- CONFIGURATION ---
SRC_URL = "uya8e9rbrxomkrmliuj8za.c0.europe-west3.gcp.weaviate.cloud"   # cluster holding the multi-tenant collection
SRC_KEY = "Z0F4OEZaUUJoYllVSk8rZV95dFh5WGhwa0pORTUreDVUVzRNV04xbTFzU0pkQ2ZoNlllNEtsb3BINU9zPV92MjAw"
# You can set TGT_URL = SRC_URL to migrate within the same instance
TGT_URL = "uya8e9rbrxomkrmliuj8za.c0.europe-west3.gcp.weaviate.cloud"
TGT_KEY = "Z0F4OEZaUUJoYllVSk8rZV95dFh5WGhwa0pORTUreDVUVzRNV04xbTFzU0pkQ2ZoNlllNEtsb3BINU9zPV92MjAw"

client_src = weaviate.connect_to_weaviate_cloud(
    cluster_url=SRC_URL,
    auth_credentials=Auth.api_key(SRC_KEY),
    additional_config=wvc.init.AdditionalConfig(timeout=wvc.init.Timeout(init=60, query=120))
)

if SRC_URL == TGT_URL:
    client_tgt = client_src
else:
    client_tgt = weaviate.connect_to_weaviate_cloud(
        cluster_url=TGT_URL,
        auth_credentials=Auth.api_key(TGT_KEY)
    )

print("Source ready:", client_src.is_ready())
print("Target ready:", client_tgt.is_ready())

Source ready: True
Target ready: True


In [ ]:
# Splits every multi-tenant (MT) collection on the SOURCE cluster into many single
# (non-MT) collections: one collection per ACTIVE tenant, named after the tenant.
# MT collections and their tenants are discovered automatically.
#
# Steps:
#   1. List source collections and keep the multi-tenant ones.
#   2. For each MT collection, list tenants and keep the active ones.
#   3. For each active tenant, create a single (non-MT) collection and copy every
#      object (properties + vectors + UUID) into it.
#
# Naming / collisions: the first tenant to claim a name gets the plain tenant name; a
# later collision is never merged -- it gets a distinct name prefixed with the source
# collection, then a numeric suffix (Acme -> Project_Acme -> Project_Acme_2 -> ...).
# Class names must match ^[A-Z][_0-9A-Za-z]*$, so names are sanitized when needed.

import logging
import re
import sys

import weaviate.classes as wvc
from weaviate.classes.config import Configure, Vectorizers
from weaviate.classes.tenants import TenantActivityStatus
from tqdm import tqdm

# --- LOGGING ---
logger = logging.getLogger("mt_to_single")
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)
    logger.setLevel(logging.INFO)
    logger.propagate = False


def to_collection_name(raw_name):
    """Sanitize a string into a valid Weaviate collection (class) name."""
    safe = re.sub(r"[^0-9A-Za-z_]", "_", raw_name)
    if not safe or not safe[0].isalpha():
        safe = "T_" + safe
    return safe[0].upper() + safe[1:]


def resolve_target_name(tenant_name, mt_name, used):
    """Return a UNIQUE target collection name. Plain tenant name first; on a collision
    create a distinct collection (never merge) prefixed with the source collection,
    then a numeric suffix until the name is free."""
    base = to_collection_name(tenant_name)
    if base != tenant_name:
        logger.warning("tenant '%s' is not a valid class name; using '%s'", tenant_name, base)

    def is_free(name):
        return name not in used and not client_tgt.collections.exists(name)

    if is_free(base):
        used.add(base)
        return base

    candidate = to_collection_name(f"{mt_name}_{tenant_name}")
    i = 2
    while not is_free(candidate):
        candidate = to_collection_name(f"{mt_name}_{tenant_name}_{i}")
        i += 1
    logger.warning("name '%s' already in use; creating '%s' instead", base, candidate)
    used.add(candidate)
    return candidate


def build_vector_config(config):
    """Re-create the vector config from a source MT collection. Extend the mapping for
    other vectorizers as needed. Client >= 4.16 uses Configure.Vectors.* with the
    vector_config= parameter (Configure.NamedVectors.* is the legacy vectorizer_config=)."""
    new_vector_config = []
    if config.vector_config:
        for vec_name, vec_data in config.vector_config.items():
            vec_type = vec_data.vectorizer.vectorizer
            logger.info("  vector '%s' -> %s", vec_name, vec_type.value)

            if vec_type == Vectorizers.TEXT2VEC_WEAVIATE:
                new_vector_config.append(
                    Configure.Vectors.text2vec_weaviate(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                        vectorize_collection_name=False,
                    )
                )
            elif vec_type == Vectorizers.TEXT2VEC_OPENAI:
                new_vector_config.append(
                    Configure.Vectors.text2vec_openai(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                    )
                )
            elif vec_type == Vectorizers.TEXT2VEC_COHERE:
                new_vector_config.append(
                    Configure.Vectors.text2vec_cohere(
                        name=vec_name,
                        source_properties=vec_data.vectorizer.source_properties,
                    )
                )
            # Add other vectorizers here if needed (Google, Jina, etc.)
            else:
                logger.warning("vectorizer '%s' not mapped; add it to build_vector_config()", vec_type.value)
    return new_vector_config


def migrate_tenant(mt_coll, mt_name, tenant_name, new_vector_config, used):
    """Copy one active tenant into its own new single (non-MT) collection."""
    target_name = resolve_target_name(tenant_name, mt_name, used)
    logger.info("tenant '%s' -> collection '%s'", tenant_name, target_name)

    # Always create a fresh single (non-MT) collection (no multi_tenancy_config).
    target_coll = client_tgt.collections.create(
        name=target_name,
        vector_config=new_vector_config,
    )

    src_tenant = mt_coll.with_tenant(tenant_name)

    count = 0
    with target_coll.batch.fixed_size(batch_size=1000) as batch:
        for obj in tqdm(src_tenant.iterator(include_vector=True), desc=target_name, unit="obj"):
            batch.add_object(
                properties=obj.properties,
                vector=obj.vector,
                uuid=obj.uuid,
            )
            count += 1

    failed = target_coll.batch.failed_objects
    if failed:
        logger.error("collection '%s': %d object(s) failed; first error: %s",
                     target_name, len(failed), failed[0])
    else:
        logger.info("collection '%s': migrated %d object(s)", target_name, count)


def migrate():
    # 1. Discover multi-tenant collections (simple=False exposes multi_tenancy_config + vector_config).
    all_configs = client_src.collections.list_all(simple=False)
    mt_names = [
        name for name, cfg in all_configs.items()
        if cfg.multi_tenancy_config and cfg.multi_tenancy_config.enabled
    ]
    if not mt_names:
        logger.warning("no multi-tenant collections found on the source cluster")
        return
    logger.info("discovered %d multi-tenant collection(s): %s", len(mt_names), mt_names)

    used = set()  # collection names already claimed during this run

    # 2. Process each MT collection.
    for mt_name in mt_names:
        logger.info("source collection '%s': replicating schema", mt_name)
        mt_coll = client_src.collections.use(mt_name)
        new_vector_config = build_vector_config(all_configs[mt_name])

        tenants = mt_coll.tenants.get()
        if not tenants:
            logger.info("source collection '%s': no tenants, skipping", mt_name)
            continue

        active_tenants = [
            name for name, t in tenants.items()
            if t.activity_status == TenantActivityStatus.ACTIVE
        ]
        logger.info("source collection '%s': %d tenant(s), %d active",
                    mt_name, len(tenants), len(active_tenants))

        # 3. Migrate each active tenant into its own single collection.
        for tenant_name in active_tenants:
            migrate_tenant(mt_coll, mt_name, tenant_name, new_vector_config, used)


try:
    migrate()
finally:
    client_src.close()
    if SRC_URL != TGT_URL:
        client_tgt.close()